# DeepOSWSRM: Super-Resolution Water Body Mapping

This notebook demonstrates the complete workflow for training and using DeepOSWSRM to generate high-resolution water maps from Sentinel-1 and Sentinel-2 imagery.

**Paper**: "Super-resolution water body mapping with a feature collaborative CNN model by fusing Sentinel-1 and Sentinel-2 images" (Yin et al., 2024)

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q earthengine-api geemap rasterio albumentations tensorboard torch torchvision

In [ ]:
# Import libraries
import ee
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Authenticate Google Earth Engine

In [ ]:
# Authenticate and initialize Earth Engine
try:
    ee.Initialize()
    print("Earth Engine already initialized")
except:
    print("Authenticating Earth Engine...")
    ee.Authenticate()
    ee.Initialize()
    print("Earth Engine initialized successfully!")

## 3. Upload Code Files

Upload the following Python files to Colab:
- `deeposwsrm_model.py`
- `data_download.py`
- `dataset.py`
- `train.py`
- `inference.py`

Or clone from GitHub if available.

In [ ]:
# If using Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy files from Drive to Colab
# !cp /content/drive/MyDrive/deeposwsrm/*.py ./

## 4. Download Training Data

In [ ]:
# Configure training sites (modify as needed)
# Edit data_download.py to customize your training sites

# Download data
!python data_download.py

In [ ]:
# Check downloaded data
!ls -lh deeposwsrm_data/
!cat deeposwsrm_data/metadata.json

## 5. Visualize Training Data

In [ ]:
import rasterio
import matplotlib.pyplot as plt

def visualize_sample(site_dir):
    """Visualize a training sample"""
    site_name = Path(site_dir).name
    
    # Load images
    s1_path = Path(site_dir) / f"{site_name}_S1.tif"
    s2_path = Path(site_dir) / f"{site_name}_S2.tif"
    mask_path = Path(site_dir) / f"{site_name}_water_mask.tif"
    
    with rasterio.open(s1_path) as src:
        s1_data = src.read()
    
    with rasterio.open(s2_path) as src:
        s2_data = src.read([3, 2, 1])  # RGB
    
    with rasterio.open(mask_path) as src:
        mask_data = src.read(1)
    
    # Plot
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    # Sentinel-1 VV
    axes[0].imshow(10*np.log10(s1_data[0]+1e-8), cmap='gray')
    axes[0].set_title('Sentinel-1 VV (dB)')
    axes[0].axis('off')
    
    # Sentinel-2 RGB
    s2_rgb = np.transpose(s2_data, (1, 2, 0))
    s2_rgb = np.clip(s2_rgb * 3, 0, 1)
    axes[1].imshow(s2_rgb)
    axes[1].set_title('Sentinel-2 RGB')
    axes[1].axis('off')
    
    # Water mask
    axes[2].imshow(mask_data, cmap='Blues')
    axes[2].set_title('Water Mask')
    axes[2].axis('off')
    
    # Overlay
    axes[3].imshow(s2_rgb)
    axes[3].imshow(mask_data, cmap='Blues', alpha=0.5)
    axes[3].set_title('RGB + Water Overlay')
    axes[3].axis('off')
    
    plt.suptitle(f'Training Sample: {site_name}')
    plt.tight_layout()
    plt.show()

# Visualize first sample
sample_dirs = list(Path('deeposwsrm_data').glob('*/'))  
if sample_dirs:
    visualize_sample(sample_dirs[0])

## 6. Train the Model

In [ ]:
# Train with scale factor 4 (default)
!python train.py \
    --data_dir ./deeposwsrm_data \
    --scale_factor 4 \
    --batch_size 4 \
    --epochs 50 \
    --learning_rate 1e-4 \
    --num_workers 2

In [ ]:
# Alternative: Train with scale factor 2 (faster, less super-resolution)
# !python train.py --data_dir ./deeposwsrm_data --scale_factor 2 --epochs 50 --batch_size 8

In [ ]:
# Load TensorBoard to monitor training
%load_ext tensorboard
%tensorboard --logdir ./outputs

## 7. Visualize Training Progress

In [ ]:
import json
import matplotlib.pyplot as plt

# Find latest training run
output_dirs = sorted(Path('./outputs').glob('scale4_*'))
if output_dirs:
    latest_dir = output_dirs[-1]
    history_file = latest_dir / 'training_history.json'
    
    if history_file.exists():
        with open(history_file) as f:
            history = json.load(f)
        
        # Plot losses
        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        
        epochs = [h['epoch'] for h in history]
        train_loss = [h['train_loss'] for h in history]
        val_loss = [h['val_loss'] for h in history]
        val_acc = [h['val_accuracy'] for h in history]
        val_iou = [h['val_iou'] for h in history]
        
        axes[0].plot(epochs, train_loss, label='Train Loss')
        axes[0].plot(epochs, val_loss, label='Val Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()
        axes[0].set_title('Training and Validation Loss')
        axes[0].grid(True)
        
        axes[1].plot(epochs, val_acc)
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].set_title('Validation Accuracy')
        axes[1].grid(True)
        
        axes[2].plot(epochs, val_iou)
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('IoU')
        axes[2].set_title('Validation IoU')
        axes[2].grid(True)
        
        plt.tight_layout()
        plt.show()
        
        print(f"Best validation loss: {min(val_loss):.4f}")
        print(f"Best validation accuracy: {max(val_acc):.4f}")
        print(f"Best validation IoU: {max(val_iou):.4f}")

## 8. Run Inference

In [ ]:
# Find best checkpoint
checkpoint_dirs = sorted(Path('./outputs').glob('scale4_*/checkpoints'))
if checkpoint_dirs:
    best_checkpoint = checkpoint_dirs[-1] / 'best.pth'
    print(f"Using checkpoint: {best_checkpoint}")
else:
    print("No checkpoint found. Please train the model first.")

In [ ]:
# Run inference on test data
# Replace with your test image paths
test_s1 = "./deeposwsrm_data/California_Reservoir/California_Reservoir_S1.tif"
test_s2 = "./deeposwsrm_data/California_Reservoir/California_Reservoir_S2.tif"
output_path = "./results/water_map.tif"

!mkdir -p results

!python inference.py \
    --checkpoint {best_checkpoint} \
    --sentinel1 {test_s1} \
    --sentinel2 {test_s2} \
    --output {output_path} \
    --visualize

## 9. Visualize Results

In [ ]:
# Display visualization
from IPython.display import Image, display

vis_path = "./results/water_map_visualization.png"
if Path(vis_path).exists():
    display(Image(vis_path))
else:
    print("Visualization not found")

In [ ]:
# Load and display results
import rasterio
import numpy as np

# Load outputs
with rasterio.open('./results/water_map.tif') as src:
    water_map = src.read(1)

with rasterio.open('./results/water_map_fraction.tif') as src:
    water_fraction = src.read(1)

with rasterio.open('./results/water_map_probability.tif') as src:
    water_prob = src.read(1)

# Display
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

im1 = axes[0].imshow(water_fraction, cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('Water Fraction (Coarse)')
axes[0].axis('off')
plt.colorbar(im1, ax=axes[0], fraction=0.046)

im2 = axes[1].imshow(water_prob, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('Water Probability (Fine)')
axes[1].axis('off')
plt.colorbar(im2, ax=axes[1], fraction=0.046)

axes[2].imshow(water_map, cmap='Blues')
axes[2].set_title('Water Map (Binary)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

# Print statistics
water_area_coarse = water_fraction.sum() * 10 * 10 / 1e6  # km²
water_area_fine = water_map.sum() * 2.5 * 2.5 / 1e6  # km² (for scale=4)

print(f"\nWater Area Statistics:")
print(f"  Coarse resolution: {water_area_coarse:.2f} km²")
print(f"  Fine resolution: {water_area_fine:.2f} km²")
print(f"  Difference: {abs(water_area_fine - water_area_coarse):.2f} km²")

## 10. Batch Processing (Optional)

In [ ]:
# Process multiple sites
from inference import WaterMapper
from pathlib import Path

# Load model
mapper = WaterMapper(str(best_checkpoint))

# Process all sites
data_dir = Path('./deeposwsrm_data')
output_dir = Path('./results/batch')
output_dir.mkdir(parents=True, exist_ok=True)

for site_dir in data_dir.glob('*/'):
    site_name = site_dir.name
    print(f"\nProcessing {site_name}...")
    
    s1_path = site_dir / f"{site_name}_S1.tif"
    s2_path = site_dir / f"{site_name}_S2.tif"
    output_path = output_dir / f"{site_name}_water_map.tif"
    
    if s1_path.exists() and s2_path.exists():
        try:
            water_fraction, water_prob, water_map = mapper.predict_from_files(
                sentinel1_path=str(s1_path),
                sentinel2_path=str(s2_path),
                output_path=str(output_path)
            )
            print(f"✓ Completed {site_name}")
        except Exception as e:
            print(f"✗ Error processing {site_name}: {e}")
    else:
        print(f"✗ Missing data for {site_name}")

print("\nBatch processing completed!")

## 11. Download Results

In [ ]:
# Zip results for download
!zip -r results.zip results/

# Download
from google.colab import files
files.download('results.zip')

## 12. Export to Google Drive (Optional)

In [ ]:
# Copy results to Google Drive
!mkdir -p /content/drive/MyDrive/DeepOSWSRM_Results
!cp -r ./results/* /content/drive/MyDrive/DeepOSWSRM_Results/
!cp -r ./outputs/* /content/drive/MyDrive/DeepOSWSRM_Results/

print("Results saved to Google Drive!")